# 04 · Modelo CNN (solo imágenes)

**Objetivo:** Entrenar una red neuronal convolucional usando únicamente las imágenes dermatoscópicas 28×28 RGB.

**Datos de entrada:** `../data/raw/hnmist_28_28_RGB.csv`, `../data/raw/HAM10000_metadata.csv`

**Resultado esperado:** Modelo guardado en `../models/cnn_model.h5` con ~74% de accuracy en test.

**Arquitectura:** 3 bloques Conv2D + MaxPooling → Flatten → Dense(256) + Dropout → Softmax(7).

# Modelo 2: CNN 
Vamos a entrenar aquí un modelo de red neuronal convolucional. Gracias a la convolución, nuestro modelo va a ser capaz de observar señales en nuestras imágenes, como pueden ser bordes, cambios de color muy bruscos... etc y así analizar patrones que se puedan dar en ellas. 
Para crear este modelo, nuestro patrón será crear una secuencia de capas donde iremos incrementando el número de neuronas (32, 64, 128...) y aplicando una capa Batchpooling entre ellas para normalizar esos datos. 
Después, tendremos que usar una capa Flatten para aplanar los datos, y a partir de ahí, Dense para tomar la decisión final y Dropout para "olvidar" las muestras anteriores y así no tener overfitting. 
En nuestra última capa densa, eso sí, utilizamos la función de activación softmax, ya que tiene que decidir entre más de 2 posibilidades
Después de una primera prueba, hemos visto que nuestro modelo empezaba a tener overfitting a partir de la epoch 10/12 aproximadamente, por lo que hemos configurado también un early stopper para que el modelo no siga entrenando y aprendiendo de las imágenes, ya que si las aprende demasiado bien, el resultado puede ser negativo, ya que respondería muy bien a las imágenes con las que ha aprendido, pero nada bien a las imágenes nuevas que podamos mostrarle 

## Carga de datos, preprocesado y entrenamiento

La CNN aprende representaciones espaciales de las imágenes directamente de los píxeles. A diferencia del modelo tabular, no requiere feature engineering manual.

**Hiperparámetros clave:**
- `batch_size=32`: balance entre estabilidad del gradiente y velocidad.
- `EarlyStopping(patience=5, restore_best_weights=True)`: detiene el entrenamiento cuando la val_loss deja de mejorar y recupera los pesos del mejor epoch, evitando overfitting.
- Filtros crecientes (32→64→128): las capas iniciales detectan bordes simples; las posteriores combinan esas features en patrones más complejos.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import matplotlib.pyplot as plt

# ── Carga y alineamiento por image_id ────────────────────────────────────────
df_images = pd.read_csv(r"../data/raw/hnmist_28_28_RGB.csv")
metadata  = pd.read_csv(r"../data/raw/HAM10000_metadata.csv")

# Garantizar mismo número de filas y asignar image_id explícitamente
n = min(len(df_images), len(metadata))
df_images = df_images.iloc[:n].reset_index(drop=True)
metadata  = metadata.iloc[:n].reset_index(drop=True)
df_images['image_id'] = metadata['image_id']

# Fusionar por clave (no por posición) para garantizar el alineamiento
df = df_images.merge(
    metadata[['image_id', 'dx']],
    on='image_id'
)

# ── Preprocesado ──────────────────────────────────────────────────────────────
pixel_cols = [c for c in df.columns if c not in ['image_id', 'dx']]
X_img = df[pixel_cols].values.astype(np.float32).reshape(-1, 28, 28, 3)
X_img /= 255.0

le = LabelEncoder()
y_encoded = le.fit_transform(df['dx'].values)
y_onehot  = to_categorical(y_encoded)
num_classes = y_onehot.shape[1]
print("Clases:", le.classes_)

# ── Split 70 / 15 / 15 ───────────────────────────────────────────────────────
X_train, X_temp, y_train, y_temp, enc_train, enc_temp = train_test_split(
    X_img, y_onehot, y_encoded,
    test_size=0.30, random_state=42, stratify=y_encoded)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50, random_state=42, stratify=enc_temp)

print(f"Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")

# ── Arquitectura CNN ──────────────────────────────────────────────────────────
input_shape = (28, 28, 3)

model = Sequential([
    Conv2D(32,  (3,3), activation='relu', padding='same', input_shape=input_shape),
    MaxPooling2D((2,2)),
    Conv2D(64,  (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),
    Conv2D(128, (3,3), activation='relu', padding='same'),
    MaxPooling2D((2,2)),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# ── Entrenamiento ─────────────────────────────────────────────────────────────
early_stop = EarlyStopping(
    monitor='val_loss', patience=10, min_delta=0.005, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=64,
    callbacks=[early_stop]
)

# ── Evaluación ────────────────────────────────────────────────────────────────
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f"Test accuracy: {test_acc:.4f}")
model.save('../models/cnn_model.h5')

plt.plot(history.history['accuracy'], label='Train')
plt.plot(history.history['val_accuracy'], label='Validación')
plt.title('Accuracy — CNN'); plt.xlabel('Epoch'); plt.ylabel('Accuracy')
plt.legend(); plt.show()

plt.plot(history.history['loss'], label='Train')
plt.plot(history.history['val_loss'], label='Validación')
plt.title('Loss — CNN'); plt.xlabel('Epoch'); plt.ylabel('Loss')
plt.legend(); plt.show()


## Evaluación detallada por clase

El accuracy global no es suficiente en diagnóstico médico. Aquí vemos cuánto acierta el modelo en **cada tipo de lesión** por separado, y lo comparamos con el **baseline ZeroR** (lo que conseguiría un modelo que predice siempre la clase más frecuente, sin aprender nada).

In [ ]:
from sklearn.metrics import classification_report
import numpy as np

# Predicciones sobre el conjunto de test
y_pred_probs = model.predict(X_test)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = np.argmax(y_test, axis=1)

# Baseline ZeroR: accuracy de predecir siempre la clase más frecuente
from collections import Counter
most_common = Counter(y_true).most_common(1)[0][1]
baseline = most_common / len(y_true)
print(f'Baseline ZeroR (predecir siempre la clase más frecuente): {baseline:.2%}')
print(f'Accuracy del modelo: {(y_pred == y_true).mean():.2%}')
print(f'Mejora sobre baseline: {(y_pred == y_true).mean() - baseline:+.2%}')

# Desglose por clase
print('
Resultados por clase diagnóstica:')
print(classification_report(y_true, y_pred, target_names=le.classes_))

# Modelo 2D, conclusiones:
Este ha sido nuestro primer modelo 2D!
Como vemos, la convolución nos ha permitido alcanzar un valor de 0.74 aprox, algo mejor que nuestro modelo tabular. Como ya expliqué en la introducción de este modelo, en el gráfico observamos que el overfitting iba a empezar a suceder a partir de estos valores. por eso, hemos configurado Earlystopping con 'restore_best_weights'=True, de manera que el modelo que tenemos corresponde a la mejor epoch donde el modelo aún aprendía. 
